# Additional Few-Shot Conditions

## Cell A — Imports and load data

In [1]:
import requests
import json
import re
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("headlines_raw.csv")
headlines = df["headline"].tolist()
true_labels = df["label"].tolist()

print(f"Headlines: {len(headlines)}")
print(df["label"].value_counts())

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=10)
    models = [m["name"] for m in r.json().get("models", [])]
    print("\nAvailable models:", models)
except Exception as e:
    print(f"Ollama not reachable: {e} — run 'ollama serve'")

Headlines: 14308
label
neutral     8990
positive    3273
negative    2045
Name: count, dtype: int64

Available models: ['safe-agent-9b:latest', 'llama3:latest', 'mistral:latest', 'safe-agent-32b:latest', 'qwen3:32b', 'qwen3.5:9b']


## Cell B — Define example banks

In [2]:
EXAMPLES_5SHOT = [
    {
        "headline": "Company reports record quarterly profit, beating analyst expectations by 15 percent.",
        "label": "positive",
        "json": '{"positive": 0.90, "negative": 0.05, "neutral": 0.05}'
    },
    {
        "headline": "Firm announces 2,000 layoffs and plant closures amid falling demand.",
        "label": "negative",
        "json": '{"positive": 0.05, "negative": 0.88, "neutral": 0.07}'
    },
    {
        "headline": "The company will publish its annual results on Thursday.",
        "label": "neutral",
        "json": '{"positive": 0.08, "negative": 0.08, "neutral": 0.84}'
    },
    {
        "headline": "Shares rose sharply after the firm raised its full-year earnings guidance.",
        "label": "positive",
        "json": '{"positive": 0.88, "negative": 0.04, "neutral": 0.08}'
    },
    {
        "headline": "Revenue declined 18 percent year-on-year due to supply chain disruptions.",
        "label": "negative",
        "json": '{"positive": 0.04, "negative": 0.90, "neutral": 0.06}'
    },
]

EXAMPLES_2SHOT_BALANCED = [
    {
        "headline": "Company reports record quarterly profit, beating analyst expectations by 15 percent.",
        "label": "positive",
        "json": '{"positive": 0.90, "negative": 0.05, "neutral": 0.05}'
    },
    {
        "headline": "Firm announces 2,000 layoffs and plant closures amid falling demand.",
        "label": "negative",
        "json": '{"positive": 0.05, "negative": 0.88, "neutral": 0.07}'
    },
    {
        "headline": "The company will publish its annual results on Thursday.",
        "label": "neutral",
        "json": '{"positive": 0.08, "negative": 0.08, "neutral": 0.84}'
    },
]

print("Balanced 2-shot examples:", len(EXAMPLES_2SHOT_BALANCED))
print("Classes:", [e["label"] for e in EXAMPLES_2SHOT_BALANCED])
print("\n5-shot examples:", len(EXAMPLES_5SHOT))
print("Classes:", [e["label"] for e in EXAMPLES_5SHOT])

Balanced 2-shot examples: 3
Classes: ['positive', 'negative', 'neutral']

5-shot examples: 5
Classes: ['positive', 'negative', 'neutral', 'positive', 'negative']


## Cell C — Shared prompt builder and parser

In [3]:
def build_fewshot_prompt(headline: str, examples: list) -> str:
    prompt = "Analyze the financial sentiment of news headlines.\n"
    prompt += 'Respond with ONLY a JSON object: {"positive": float, "negative": float, "neutral": float}\n'
    prompt += "All three values must sum to 1.0. No explanation, just the JSON.\n\n"
    prompt += "Examples:\n"
    for ex in examples:
        prompt += f"Headline: {ex['headline']}\n"
        prompt += f"Output: {ex['json']}\n\n"
    prompt += f"Headline: {headline}\n"
    prompt += "Output:"
    return prompt


def parse_json_sentiment(text: str) -> dict:
    default = {"positive": 0.3334, "negative": 0.3333, "neutral": 0.3333}
    try:
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        match = re.search(r'\{[^{}]+\}', text)
        if match:
            result = json.loads(match.group())
            if all(k in result for k in ['positive', 'negative', 'neutral']):
                total = sum(result.values())
                if total > 0:
                    result = {k: v / total for k, v in result.items()}
                return result
    except Exception:
        pass
    return default


def get_fewshot_sentiment(headline: str, model: str, examples: list,
                           think: bool = False) -> dict:
    prompt = build_fewshot_prompt(headline, examples)
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 120}
    }
    if not think:
        payload["think"] = False
    try:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json=payload,
            timeout=90
        )
        text = r.json().get("response", "")
        return parse_json_sentiment(text)
    except Exception:
        return {"positive": 0.3334, "negative": 0.3333, "neutral": 0.3333}

print("=== SANITY CHECK ===")
for h in headlines[:2]:
    r = get_fewshot_sentiment(h, "mistral:latest", EXAMPLES_2SHOT_BALANCED, think=True)
    print(f"Mistral balanced-2shot: {r}  → {max(r, key=r.get)}")
    r = get_fewshot_sentiment(h, "qwen3.5:9b", EXAMPLES_5SHOT, think=False)
    print(f"Qwen3.5 5shot:          {r}  → {max(r, key=r.get)}")
    print()

=== SANITY CHECK ===
Mistral balanced-2shot: {'positive': 0.65, 'negative': 0.15, 'neutral': 0.2}  → positive
Qwen3.5 5shot:          {'positive': 0.15, 'negative': 0.15, 'neutral': 0.7}  → neutral

Mistral balanced-2shot: {'positive': 0.95, 'negative': 0.02, 'neutral': 0.03}  → positive
Qwen3.5 5shot:          {'positive': 0.15, 'negative': 0.05, 'neutral': 0.8}  → neutral



## Cell D — Run Mistral-7B balanced 2-shot

In [4]:
SAVE_PATH_MB2 = "fewshot_mistral_2shot_balanced.csv"

if os.path.exists(SAVE_PATH_MB2):
    done = pd.read_csv(SAVE_PATH_MB2)
    start_idx = len(done)
    results = done.to_dict("records")
    print(f"Resuming Mistral balanced-2shot from index {start_idx} / {len(headlines)}")
else:
    results = []
    start_idx = 0
    print(f"Starting fresh. Total: {len(headlines)} headlines")

for i, h in enumerate(
    tqdm(headlines[start_idx:], desc="Mistral balanced-2shot"), start=start_idx
):
    s = get_fewshot_sentiment(h, "mistral:latest", EXAMPLES_2SHOT_BALANCED, think=True)
    results.append({
        "mistral_2shot_bal_positive": s["positive"],
        "mistral_2shot_bal_negative": s["negative"],
        "mistral_2shot_bal_neutral":  s["neutral"]
    })
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH_MB2, index=False)

pd.DataFrame(results).to_csv(SAVE_PATH_MB2, index=False)
print(f"Done. Saved to {SAVE_PATH_MB2}")

Starting fresh. Total: 14308 headlines


Mistral balanced-2shot: 100%|███████████████████████████████████████████████████| 14308/14308 [2:18:57<00:00,  1.72it/s]

Done. Saved to fewshot_mistral_2shot_balanced.csv


## Cell E — Run Qwen3.5-9B 2-shot

In [5]:
SAVE_PATH_Q2 = "fewshot_qwen3_2shot.csv"

if os.path.exists(SAVE_PATH_Q2):
    done = pd.read_csv(SAVE_PATH_Q2)
    start_idx = len(done)
    results = done.to_dict("records")
    print(f"Resuming Qwen3.5 2-shot from index {start_idx} / {len(headlines)}")
else:
    results = []
    start_idx = 0
    print(f"Starting fresh. Total: {len(headlines)} headlines")

for i, h in enumerate(
    tqdm(headlines[start_idx:], desc="Qwen3.5 2-shot"), start=start_idx
):
    s = get_fewshot_sentiment(h, "qwen3.5:9b", EXAMPLES_2SHOT_BALANCED, think=False)
    results.append({
        "qwen3_2shot_positive": s["positive"],
        "qwen3_2shot_negative": s["negative"],
        "qwen3_2shot_neutral":  s["neutral"]
    })
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH_Q2, index=False)

pd.DataFrame(results).to_csv(SAVE_PATH_Q2, index=False)
print(f"Done. Saved to {SAVE_PATH_Q2}")

Starting fresh. Total: 14308 headlines


Qwen3.5 2-shot:  74%|██████████████████████████████████████████▍              | 10658/14308 [4:21:27<1:29:40,  1.47s/it]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Qwen3.5 2-shot: 100%|███████████████████████████████████████████████████████████| 14308/14308 [5:50:45<00:00,  1.47s/it]

Done. Saved to fewshot_qwen3_2shot.csv


## Cell F — Run Qwen3.5-9B 5-shot

In [6]:
SAVE_PATH_Q5 = "fewshot_qwen3_5shot.csv"

if os.path.exists(SAVE_PATH_Q5):
    done = pd.read_csv(SAVE_PATH_Q5)
    start_idx = len(done)
    results = done.to_dict("records")
    print(f"Resuming Qwen3.5 5-shot from index {start_idx} / {len(headlines)}")
else:
    results = []
    start_idx = 0
    print(f"Starting fresh. Total: {len(headlines)} headlines")

for i, h in enumerate(
    tqdm(headlines[start_idx:], desc="Qwen3.5 5-shot"), start=start_idx
):
    s = get_fewshot_sentiment(h, "qwen3.5:9b", EXAMPLES_5SHOT, think=False)
    results.append({
        "qwen3_5shot_positive": s["positive"],
        "qwen3_5shot_negative": s["negative"],
        "qwen3_5shot_neutral":  s["neutral"]
    })
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH_Q5, index=False)

pd.DataFrame(results).to_csv(SAVE_PATH_Q5, index=False)
print(f"Done. Saved to {SAVE_PATH_Q5}")

Starting fresh. Total: 14308 headlines


Qwen3.5 5-shot: 100%|███████████████████████████████████████████████████████████| 14308/14308 [5:59:11<00:00,  1.51s/it]

Done. Saved to fewshot_qwen3_5shot.csv


## Cell G — Evaluate all three new conditions

In [7]:
df_base  = pd.read_csv("headlines_raw.csv")
df_mb2   = pd.read_csv("fewshot_mistral_2shot_balanced.csv")
df_q2    = pd.read_csv("fewshot_qwen3_2shot.csv")
df_q5    = pd.read_csv("fewshot_qwen3_5shot.csv")

df_q0    = pd.read_csv("sentiment_qwen3.csv")

df_new = pd.concat([
    df_base.reset_index(drop=True),
    df_mb2.reset_index(drop=True),
    df_q2.reset_index(drop=True),
    df_q5.reset_index(drop=True),
    df_q0.reset_index(drop=True),
], axis=1)

true = df_new["label"].tolist()

def probs_to_label(row, pos_col, neg_col, neu_col):
    scores = {
        "positive": row[pos_col],
        "negative": row[neg_col],
        "neutral":  row[neu_col]
    }
    return max(scores, key=scores.get)

conditions = [
    ("Mistral-7B (2-shot balanced)",
     "mistral_2shot_bal_positive", "mistral_2shot_bal_negative", "mistral_2shot_bal_neutral"),
    ("Qwen3.5-9B (zero-shot)",
     "qwen3_positive",    "qwen3_negative",    "qwen3_neutral"),
    ("Qwen3.5-9B (2-shot)",
     "qwen3_2shot_positive", "qwen3_2shot_negative", "qwen3_2shot_neutral"),
    ("Qwen3.5-9B (5-shot)",
     "qwen3_5shot_positive", "qwen3_5shot_negative", "qwen3_5shot_neutral"),
]

print(f"{'Model / Condition':<35} {'Accuracy':>9} {'F1_macro':>9} {'Neu_F1':>8}")
print("-" * 65)

rows = []
for name, pc, nc, nuc in conditions:
    pred = df_new.apply(lambda r: probs_to_label(r, pc, nc, nuc), axis=1).tolist()
    acc  = accuracy_score(true, pred)
    f1m  = f1_score(true, pred, average="macro",
                    labels=["positive", "negative", "neutral"])
    neu_f1 = f1_score(true, pred, average=None,
                      labels=["negative", "neutral", "positive"])[1]

    pb_mask = df_new["source"] == "phrasebank"
    tw_mask = df_new["source"] == "twitter"
    acc_pb  = accuracy_score(
        df_new.loc[pb_mask, "label"],
        [p for p, m in zip(pred, pb_mask) if m]
    )
    acc_tw  = accuracy_score(
        df_new.loc[tw_mask, "label"],
        [p for p, m in zip(pred, tw_mask) if m]
    )

    df_new[f"{name}_pred"] = pred
    print(f"{name:<35} {acc:>9.4f} {f1m:>9.4f} {neu_f1:>8.4f}")
    rows.append({
        "Model": name, "Accuracy": acc, "F1_macro": f1m,
        "F1_weighted": f1_score(true, pred, average="weighted",
                                labels=["positive","negative","neutral"]),
        "Acc_PhraseBank": acc_pb, "Acc_Twitter": acc_tw,
        "Neutral_F1": neu_f1
    })

df_results_new = pd.DataFrame(rows)
df_results_new.to_csv("fewshot_additional_evaluation.csv", index=False)

Model / Condition                    Accuracy  F1_macro   Neu_F1
-----------------------------------------------------------------
Mistral-7B (2-shot balanced)           0.3929    0.4192   0.1212
Qwen3.5-9B (zero-shot)                 0.6982    0.6928   0.7141
Qwen3.5-9B (2-shot)                    0.7520    0.7217   0.7987
Qwen3.5-9B (5-shot)                    0.7607    0.7260   0.8093


## Cell H — The key question: Mistral balanced 2-shot vs original 2-shot

In [9]:
df_orig = pd.read_csv("fewshot_results.csv")
true_arr = np.array(true)

orig_m2_pred = df_orig.apply(
    lambda r: max(
        {"positive": r["mistral_2shot_positive"],
         "negative": r["mistral_2shot_negative"],
         "neutral":  r["mistral_2shot_neutral"]},
        key=lambda k: {"positive": r["mistral_2shot_positive"],
                       "negative": r["mistral_2shot_negative"],
                       "neutral":  r["mistral_2shot_neutral"]}[k]
    ), axis=1
).tolist()

bal_m2_pred = df_new["Mistral-7B (2-shot balanced)_pred"].tolist()

orig_acc   = accuracy_score(true, orig_m2_pred)
bal_acc    = accuracy_score(true, bal_m2_pred)
orig_neu   = f1_score(true, orig_m2_pred, average=None,
                      labels=["negative","neutral","positive"])[1]
bal_neu    = f1_score(true, bal_m2_pred, average=None,
                      labels=["negative","neutral","positive"])[1]

print("=== MISTRAL 2-SHOT: POLAR-ONLY vs BALANCED ===")
print(f"{'Condition':<30} {'Accuracy':>10} {'Neutral F1':>12}")
print("-" * 55)
print(f"{'Original 2-shot (polar only)':<30} {orig_acc:>10.4f} {orig_neu:>12.4f}")
print(f"{'Balanced 2-shot (+neutral ex)':<30} {bal_acc:>10.4f} {bal_neu:>12.4f}")

=== MISTRAL 2-SHOT: POLAR-ONLY vs BALANCED ===
Condition                        Accuracy   Neutral F1
-------------------------------------------------------
Original 2-shot (polar only)       0.4179       0.1996
Balanced 2-shot (+neutral ex)      0.3929       0.1212

INTERPRETATION: Mistral STILL degrades with balanced examples.
→ Degradation is a MODEL PROPERTY, not an example-selection artefact.
→ This STRENGTHENS the paper's claim in Section 4.6.


## Cell I — Neutral class distribution for new conditions

In [10]:
neutral_df = df_new[df_new["label"] == "neutral"]
n = len(neutral_df)
print(f"Truly neutral headlines: {n}\n")

new_conditions = [
    ("Mistral balanced-2shot",  "Mistral-7B (2-shot balanced)_pred"),
    ("Qwen3.5 zero-shot",       "Qwen3.5-9B (zero-shot)_pred"),
    ("Qwen3.5 2-shot",          "Qwen3.5-9B (2-shot)_pred"),
    ("Qwen3.5 5-shot",          "Qwen3.5-9B (5-shot)_pred"),
]

print(f"{'Condition':<28} | {'Neg':>6} {'Neu':>7} {'Pos':>7} | {'Neu F1':>8}")
print("-" * 65)
for name, col in new_conditions:
    counts = neutral_df[col].value_counts(normalize=True) * 100
    neg = counts.get("negative", 0)
    neu = counts.get("neutral",  0)
    pos = counts.get("positive", 0)
    # Compute neutral F1 over full dataset
    pred_full = df_new[col].tolist()
    nf1 = f1_score(true, pred_full, average=None,
                   labels=["negative","neutral","positive"])[1]
    print(f"{name:<28} | {neg:>5.1f}% {neu:>6.1f}% {pos:>6.1f}% | {nf1:>8.4f}")

Truly neutral headlines: 8990

Condition                    |    Neg     Neu     Pos |   Neu F1
-----------------------------------------------------------------
Mistral balanced-2shot       |  18.3%    6.5%   75.3% |   0.1212
Qwen3.5 zero-shot            |  17.0%   58.4%   24.6% |   0.7141
Qwen3.5 2-shot               |  13.7%   76.2%   10.1% |   0.7987
Qwen3.5 5-shot               |  12.6%   79.0%    8.4% |   0.8093


## Cell J — McNemar tests: new conditions vs zero-shot baselines

In [11]:
from statsmodels.stats.contingency_tables import mcnemar

mistral_0shot_pred = df_orig.apply(
    lambda r: max(
        {"positive": r["mistral_positive"],
         "negative": r["mistral_negative"],
         "neutral":  r["mistral_neutral"]},
        key=lambda k: {"positive": r["mistral_positive"],
                       "negative": r["mistral_negative"],
                       "neutral":  r["mistral_neutral"]}[k]
    ), axis=1
).tolist()

qwen3_0shot_pred = df_new["Qwen3.5-9B (zero-shot)_pred"].tolist()

def mcnemar_test(pred_a, pred_b):
    a = np.array(pred_a)
    b = np.array(pred_b)
    t = np.array(true)
    n10 = ((a == t) & (b != t)).sum()
    n01 = ((a != t) & (b == t)).sum()
    result = mcnemar([[0, n10], [n01, 0]], exact=False, correction=True)
    return result.pvalue

tests = [
    ("Mistral 0-shot vs balanced-2shot",
     mistral_0shot_pred,
     df_new["Mistral-7B (2-shot balanced)_pred"].tolist()),
    ("Qwen3.5 0-shot vs 2-shot",
     qwen3_0shot_pred,
     df_new["Qwen3.5-9B (2-shot)_pred"].tolist()),
    ("Qwen3.5 0-shot vs 5-shot",
     qwen3_0shot_pred,
     df_new["Qwen3.5-9B (5-shot)_pred"].tolist()),
    ("Qwen3.5 2-shot vs 5-shot",
     df_new["Qwen3.5-9B (2-shot)_pred"].tolist(),
     df_new["Qwen3.5-9B (5-shot)_pred"].tolist()),
]

print(f"{'Comparison':<40} {'p-value':>10} {'Significant?'}")
print("-" * 65)
for label, a, b in tests:
    p = mcnemar_test(a, b)
    sig = "YES (p<0.05)" if p < 0.05 else "NO"
    print(f"{label:<40} {p:>10.4f} {sig}")

Comparison                                  p-value Significant?
-----------------------------------------------------------------
Mistral 0-shot vs balanced-2shot             0.0000 YES (p<0.05)
Qwen3.5 0-shot vs 2-shot                     0.0000 YES (p<0.05)
Qwen3.5 0-shot vs 5-shot                     0.0000 YES (p<0.05)
Qwen3.5 2-shot vs 5-shot                     0.0000 YES (p<0.05)


## Cell K — Final expanded table

In [13]:
df_orig_eval = pd.read_csv("fewshot_evaluation.csv")

df_combined = pd.concat([df_orig_eval, df_results_new], ignore_index=True)

for col in ["Accuracy", "F1_macro", "F1_weighted", "Acc_PhraseBank", "Acc_Twitter"]:
    if col in df_combined.columns:
        df_combined[col] = df_combined[col].map(lambda x: f"{x:.4f}")

print("=== FULL EXPANDED RESULTS TABLE ===")
print(df_combined[["Model", "Accuracy", "F1_macro",
                   "Acc_PhraseBank", "Acc_Twitter"]].to_string(index=False))

df_combined.to_csv("fewshot_table_expanded.csv", index=False)
print("\nSaved: fewshot_table_expanded.csv")

=== FULL EXPANDED RESULTS TABLE ===
                       Model Accuracy F1_macro Acc_PhraseBank Acc_Twitter
         FinBERT (zero-shot)   0.7712   0.7373         0.8890      0.7113
      Mistral-7B (zero-shot)   0.5347   0.5419         0.6471      0.4775
         Mistral-7B (2-shot)   0.4179   0.4493         0.4684      0.3921
         Mistral-7B (5-shot)   0.4145   0.4354         0.4572      0.3927
       Llama3-8B (zero-shot)   0.3502   0.3768         0.3932      0.3283
          Llama3-8B (2-shot)   0.4373   0.4477         0.5560      0.3768
          Llama3-8B (5-shot)   0.5319   0.5386         0.6985      0.4471
Mistral-7B (2-shot balanced)   0.3929   0.4192         0.4351      0.3713
      Qwen3.5-9B (zero-shot)   0.6982   0.6928         0.8101      0.6412
         Qwen3.5-9B (2-shot)   0.7520   0.7217         0.8080      0.7235
         Qwen3.5-9B (5-shot)   0.7607   0.7260         0.7971      0.7422

Saved: fewshot_table_expanded.csv


In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

df_base  = pd.read_csv("headlines_raw.csv")
df_mb2   = pd.read_csv("fewshot_mistral_2shot_balanced.csv")
df_q2    = pd.read_csv("fewshot_qwen3_2shot.csv")
df_q5    = pd.read_csv("fewshot_qwen3_5shot.csv")

true = df_base["label"].tolist()

def probs_to_label(pos, neg, neu):
    scores = {"positive": pos, "negative": neg, "neutral": neu}
    return max(scores, key=scores.get)

pred_mb2 = [probs_to_label(r["mistral_2shot_bal_positive"],
                            r["mistral_2shot_bal_negative"],
                            r["mistral_2shot_bal_neutral"])
            for _, r in df_mb2.iterrows()]

pred_q2  = [probs_to_label(r["qwen3_2shot_positive"],
                            r["qwen3_2shot_negative"],
                            r["qwen3_2shot_neutral"])
            for _, r in df_q2.iterrows()]

pred_q5  = [probs_to_label(r["qwen3_5shot_positive"],
                            r["qwen3_5shot_negative"],
                            r["qwen3_5shot_neutral"])
            for _, r in df_q5.iterrows()]

labels = ["positive", "negative", "neutral"]

results = {
    "Mistral-7B (2-shot balanced)": pred_mb2,
    "Qwen3.5-9B (2-shot)":          pred_q2,
    "Qwen3.5-9B (5-shot)":          pred_q5,
}

print(f"{'Condition':<35} {'F1_weighted':>12} {'F1_macro':>10} {'Accuracy':>10}")
print("-" * 70)

output_rows = []
for name, pred in results.items():
    f1w = f1_score(true, pred, average="weighted", labels=labels)
    f1m = f1_score(true, pred, average="macro",    labels=labels)
    acc = sum(p == t for p, t in zip(pred, true)) / len(true)
    print(f"{name:<35} {f1w:>12.4f} {f1m:>10.4f} {acc:>10.4f}")
    output_rows.append({
        "Condition": name,
        "F1_weighted": round(f1w, 4),
        "F1_macro":    round(f1m, 4),
        "Accuracy":    round(acc, 4)
    })

df_out = pd.DataFrame(output_rows)
df_out.to_csv("fewshot_f1weighted_additional.csv", index=False)
print("\nSaved to fewshot_f1weighted_additional.csv")

Condition                            F1_weighted   F1_macro   Accuracy
----------------------------------------------------------------------
Mistral-7B (2-shot balanced)              0.2793     0.4192     0.3929
Qwen3.5-9B (2-shot)                       0.7548     0.7217     0.7520
Qwen3.5-9B (5-shot)                       0.7622     0.7260     0.7607

Saved to fewshot_f1weighted_additional.csv
